# Cosmos DB Mirrored Data Transformation

This notebook processes the `conversations` and `interactions` tables mirrored from the
Challenge 1 Cosmos DB `agentsdb` database into the `CosmosDB-agentsdb` Fabric item.
The two mirrored tables must be exposed in the default `Observability` Lakehouse as
OneLake table shortcuts named `conversations` and `interactions`.

It reads the shortcut-backed Delta tables, conforms the camelCase Cosmos document fields,
sessionizes interactions, and writes Silver and Gold analytics tables.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    DoubleType, TimestampType, DateType, DecimalType, ArrayType, BooleanType
)


## Read Mirrored Table Shortcuts

Microsoft Fabric notebooks access Mirrored Database tables through OneLake table shortcuts
in the default Lakehouse. These shortcuts target `CosmosDB-agentsdb` and retain the source
container names `conversations` and `interactions`.

**Prerequisite — create the shortcuts before running the next cell.** They are not created by
`setup_fabric_workspace.py` or `setup_cosmos_mirroring.py`. Once Mirroring reports a `running`
status, in the `Observability` Lakehouse:

1. Hover **Tables**, then choose **New shortcut → Microsoft OneLake**.
2. Select the `CosmosDB-agentsdb` mirrored database as the data source.
3. Expand `Tables` → `agentsdb` and tick both `conversations` and `interactions`.
4. Create the shortcuts and confirm they land at `Tables/dbo/conversations` and
   `Tables/dbo/interactions`.

If the next cell raises a path-not-found error, the shortcuts are missing or Mirroring has not
yet materialised the source tables.


In [ ]:
MIRRORED_DATABASE_ITEM = "CosmosDB-agentsdb"
CONVERSATIONS_SHORTCUT = "Tables/dbo/conversations"
INTERACTIONS_SHORTCUT = "Tables/dbo/interactions"

raw_conversations = spark.read.format("delta").load(CONVERSATIONS_SHORTCUT)
raw_interactions = spark.read.format("delta").load(INTERACTIONS_SHORTCUT)

print(f"Mirrored database item: {MIRRORED_DATABASE_ITEM}")
print(f"Raw conversations: {raw_conversations.count()} rows")
print(f"Raw interactions:  {raw_interactions.count()} rows")


## Conform Mirrored Cosmos Documents

The Challenge 1 workload writes camelCase Cosmos properties. Mirroring exposes top-level
properties as columns and nested `metadata` as JSON. The following cells preserve the
actual conversation and interaction fields without inventing unavailable satisfaction,
participant, or processing-time data.


In [ ]:
metadata_schema = StructType([
    StructField("source", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("language", StringType(), True),
    StructField("client_version", StringType(), True),
    StructField("tags", ArrayType(StringType()), True)
])

silver_conversations = (
    raw_conversations
    .withColumn("metadata_parsed", F.from_json(F.col("metadata"), metadata_schema))
    .select(
        F.col("id").cast(StringType()).alias("conversation_id"),
        F.col("sessionId").cast(StringType()).alias("session_id"),
        F.col("title").cast(StringType()),
        F.col("createdAt").cast(TimestampType()).alias("conversation_started_at"),
        F.col("updatedAt").cast(TimestampType()).alias("conversation_updated_at"),
        F.col("metadata_parsed.source").alias("source_system"),
        F.col("metadata_parsed.channel").alias("channel"),
        F.col("metadata_parsed.language").alias("language"),
        F.col("metadata_parsed.client_version").alias("client_version"),
        F.col("metadata_parsed.tags").alias("tags")
    )
    .withColumn("conversation_date", F.col("conversation_started_at").cast(DateType()))
)

print(f"Silver conversations prepared - {silver_conversations.count()} rows")


In [ ]:
silver_interactions = (
    raw_interactions
    .select(
        F.col("id").cast(StringType()).alias("interaction_id"),
        F.col("conversationId").cast(StringType()).alias("conversation_id"),
        F.col("sessionId").cast(StringType()).alias("session_id"),
        F.col("role").cast(StringType()).alias("interaction_type"),
        F.col("timestamp").cast(TimestampType()).alias("interaction_timestamp"),
        F.when(F.col("role") == "user", F.col("content")).cast(StringType()).alias("user_message"),
        F.when(F.col("role") == "assistant", F.col("content")).cast(StringType()).alias("agent_response"),
        F.col("durationMs").cast(DoubleType()).alias("response_time_ms"),
        F.col("model").cast(StringType()).alias("model_name"),
        F.col("promptTokens").cast(LongType()).alias("prompt_tokens"),
        F.col("completionTokens").cast(LongType()).alias("completion_tokens"),
        F.col("totalTokens").cast(LongType()).alias("total_tokens"),
        F.col("status").cast(StringType()).alias("interaction_status")
    )
    .withColumn("interaction_date", F.col("interaction_timestamp").cast(DateType()))
)

print(f"Silver interactions prepared - {silver_interactions.count()} rows")


## Sessionize Interactions

Group interactions into sessions and compute session-level boundaries using `lag` and
`lead` window functions. This enriches each interaction row with its ordinal position
within the session and flags session start/end boundaries.


In [ ]:
session_window = (
    Window
    .partitionBy("session_id")
    .orderBy("interaction_timestamp")
)

silver_interactions_sessionized = (
    silver_interactions
    .withColumn("prev_interaction_ts", F.lag("interaction_timestamp").over(session_window))
    .withColumn("next_interaction_ts", F.lead("interaction_timestamp").over(session_window))
    .withColumn("interaction_order", F.row_number().over(session_window))
    .withColumn("is_session_start", F.col("prev_interaction_ts").isNull())
    .withColumn("is_session_end", F.col("next_interaction_ts").isNull())
    .withColumn("time_since_prev_ms",
        F.when(
            F.col("prev_interaction_ts").isNotNull(),
            (F.unix_timestamp("interaction_timestamp") - F.unix_timestamp("prev_interaction_ts")) * 1000
        ).otherwise(F.lit(0))
    )
    .drop("prev_interaction_ts", "next_interaction_ts")
)

print(f"Sessionized interactions — {silver_interactions_sessionized.count()} rows")


## Write Silver Tables

Persist the cleansed conversation and interaction data as Delta tables for
downstream Gold aggregation and ad-hoc analysis.


In [ ]:
(
    silver_conversations
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("Tables/dbo/silver_conversations")
)
print("silver_conversations written")

(
    silver_interactions_sessionized
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("Tables/dbo/silver_interactions")
)
print("silver_interactions written")


## Gold Agent Analytics

Aggregate interaction data into a daily model-performance summary:
- Average response time reported by the agent workload
- Total prompt, completion, and combined token usage
- Conversation, interaction, and session counts
- Topic classification using keyword matching


In [ ]:
interactions_with_topic = (
    silver_interactions_sessionized
    .withColumn("topic_classification",
        F.when(F.lower(F.col("user_message")).contains("error") | F.lower(F.col("user_message")).contains("fail") | F.lower(F.col("user_message")).contains("broken"), "troubleshooting")
         .when(F.lower(F.col("user_message")).contains("how") | F.lower(F.col("user_message")).contains("help") | F.lower(F.col("user_message")).contains("guide"), "how-to")
         .when(F.lower(F.col("user_message")).contains("cost") | F.lower(F.col("user_message")).contains("price") | F.lower(F.col("user_message")).contains("billing"), "billing")
         .when(F.lower(F.col("user_message")).contains("create") | F.lower(F.col("user_message")).contains("deploy") | F.lower(F.col("user_message")).contains("provision"), "provisioning")
         .when(F.lower(F.col("user_message")).contains("permission") | F.lower(F.col("user_message")).contains("access") | F.lower(F.col("user_message")).contains("role"), "access-management")
         .otherwise("general")
    )
)

session_stats = (
    interactions_with_topic
    .groupBy("session_id")
    .agg(
        F.count("*").alias("messages_per_session"),
        F.min("interaction_timestamp").alias("session_start"),
        F.max("interaction_timestamp").alias("session_end")
    )
    .withColumn("session_duration_seconds",
        (F.unix_timestamp("session_end") - F.unix_timestamp("session_start")).cast(LongType())
    )
)

gold_agent_analytics = (
    interactions_with_topic
    .groupBy("model_name", "interaction_date", "topic_classification")
    .agg(
        F.avg("response_time_ms").cast(DecimalType(12, 2)).alias("avg_response_time_ms"),
        F.sum("prompt_tokens").alias("total_prompt_tokens"),
        F.sum("completion_tokens").alias("total_completion_tokens"),
        F.sum("total_tokens").alias("total_tokens"),
        F.countDistinct("conversation_id").alias("conversation_count"),
        F.count("*").alias("interaction_count"),
        F.countDistinct("session_id").alias("session_count")
    )
    .withColumn("interaction_date", F.col("interaction_date").cast(DateType()))
)

(
    gold_agent_analytics
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save("Tables/dbo/gold_agent_analytics")
)

print(f"gold_agent_analytics written - {gold_agent_analytics.count()} rows")
print("\nCosmos DB mirrored data transformation complete.")
